In [2]:
library(tidyverse)
library(MatchIt)
library(broom)
library(readr)
library(tibble)

## patent

In [3]:
file_path <- "/home/bingkzhao2/patent_pwords/Bingkun/Data/patent_reg_data.csv"
df <- read.csv(file_path, stringsAsFactors = FALSE)

In [4]:
df_clean <- df %>%
  mutate(

    hype_quintile = ntile(frac_hype_words, 5),

    team_gender_n = as.factor(team_gender),
    attorney_gender_n = as.factor(attorney_gender),
    examiner_gender_n = as.factor(examiner_gender),
    cpc_class_n = as.factor(cpc_class),
    cpc_section_n = as.factor(cpc_section),
    applicant_entity_n = as.factor(applicant_entity),
    business_entity_status_n = as.factor(business_entity_status),
    year = as.factor(year), 
    
    num_words_log = log(num_words + 1),
    num_prior_apps_log = log(num_prior_apps + 1),
    num_prior_patents_log = log(num_prior_patents + 1),
    attorney_num_prior_apps_log = log(attorney_num_prior_apps + 1),
    examiner_num_prior_apps_log = log(examiner_num_prior_apps + 1),
    attorney_num_prior_patents_log = log(attorney_num_prior_patents + 1),
    
    num_words_5cate = as.factor(ntile(num_words, 5)),
    examiner_num_prior_apps_5cate = as.factor(ntile(examiner_num_prior_apps, 5)),

    novelty_min_qnorm = percent_rank(novelty_tail_10)
  )

In [5]:
psm_covariates_formula <- as.formula("is_Q5 ~ flesch_reading_ease + concreteness + 
                                     novelty_min_qnorm + num_cpc_code + num_inventors + 
                                     num_words_5cate + examiner_num_prior_apps_5cate + 
                                     num_prior_apps_log + num_prior_patents_log + 
                                     attorney_num_prior_apps_log + 
                                     team_gender_n + attorney_gender_n + examiner_gender_n + 
                                     year + cpc_class_n + applicant_entity_n + business_entity_status_n")

In [6]:
outcome_formula <- as.formula("decision ~ is_Q5 + flesch_reading_ease + concreteness + 
                                     novelty_min_qnorm + num_cpc_code + num_inventors + 
                                     num_words_5cate + examiner_num_prior_apps_5cate + 
                                     num_prior_apps_log + num_prior_patents_log + 
                                     attorney_num_prior_apps_log + 
                                     team_gender_n + attorney_gender_n + examiner_gender_n + 
                                     year + cpc_class_n + applicant_entity_n + business_entity_status_n")

### PSM

In [8]:
results_table <- data.frame()
comparison_groups <- c(1, 2, 3, 4)

for (comp_group in comparison_groups) {
  
  message(paste0("--------------------------------------------------"))
  message(paste0("Computing: Q", comp_group, " (Control) vs Q5 (Treated)..."))

  df_sub <- df_clean %>%
    filter(hype_quintile %in% c(comp_group, 5)) %>%
    mutate(
      is_Q5 = ifelse(hype_quintile == 5, 1, 0)
    )
  
  m.out <- matchit(psm_covariates_formula,
                   data = df_sub,
                   method = "nearest", 
                   ratio = 1,
                   replace = FALSE)
  
  df_matched <- match.data(m.out)
  q5_outcomes <- df_matched$decision[df_matched$is_Q5 == 1]
  control_outcomes <- df_matched$decision[df_matched$is_Q5 == 0]

  t_res <- t.test(x = q5_outcomes, y = control_outcomes)

  diff_val <- t_res$estimate[2] - t_res$estimate[1]
  ci_lower_flipped <- t_res$conf.int[2] * -1
  ci_upper_flipped <- t_res$conf.int[1] * -1

  tidy_res <- data.frame(
    term = "is_Q5",
    estimate = diff_val,             
    std.error = t_res$stderr,        
    p.value = t_res$p.value,         
    conf.low = ci_lower_flipped,    
    conf.high = ci_upper_flipped,   
    comparison = paste0("Q", comp_group, " vs Q5"),
    group_id = comp_group
  )
  
  results_table <- bind_rows(results_table, tidy_res)
}

print(results_table %>% select(comparison, estimate, p.value, conf.low, conf.high))

--------------------------------------------------

Computing: Q1 (Control) vs Q5 (Treated)...

Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
--------------------------------------------------

Computing: Q2 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q3 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q4 (Control) vs Q5 (Treated)...



              comparison   estimate       p.value   conf.low  conf.high
mean of y...1   Q1 vs Q5 0.05564539  0.000000e+00 0.05389496 0.05739582
mean of y...2   Q2 vs Q5 0.05847013  0.000000e+00 0.05672198 0.06021828
mean of y...3   Q3 vs Q5 0.05073620  0.000000e+00 0.04898188 0.05249052
mean of y...4   Q4 vs Q5 0.03463536 5.928788e-323 0.03286903 0.03640169


### Balance Check

In [29]:
balance_table <- data.frame()
comparison_groups <- c(1, 2, 3, 4)

for (comp_group in comparison_groups) {
  
  message(paste0("--------------------------------------------------"))
  message(paste0("Computing: Q", comp_group, " (Control) vs Q5 (Treated)..."))

  df_sub <- df_clean %>%
    filter(hype_quintile %in% c(comp_group, 5)) %>%
    mutate(
      is_Q5 = ifelse(hype_quintile == 5, 1, 0)
    )
  
  m.out <- matchit(psm_covariates_formula,
                   data = df_sub,
                   method = "nearest", 
                   distance = "glm",  
                   caliper = 0.05,    
                   ratio = 1,
                   replace = FALSE)
  
  balance_summary <- summary(m.out, un = TRUE) 
  

  tidy_balance <- as.data.frame(balance_summary$sum.matched) %>%
    rownames_to_column(var = "covariate") %>%               
    select(covariate, matched_SMD = `Std. Mean Diff.`) %>%
    mutate(
      comparison = paste0("Q", comp_group, " vs Q5"),     
      group_id = comp_group                                
    )
  balance_table <- bind_rows(balance_table, tidy_balance)
}
print(balance_table)

--------------------------------------------------

Computing: Q1 (Control) vs Q5 (Treated)...

Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
--------------------------------------------------

Computing: Q2 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q3 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q4 (Control) vs Q5 (Treated)...



                               covariate   matched_SMD comparison group_id
1                               distance  4.097615e-02   Q1 vs Q5        1
2                    flesch_reading_ease -2.336098e-02   Q1 vs Q5        1
3                           concreteness  6.826595e-03   Q1 vs Q5        1
4                      novelty_min_qnorm -2.624129e-02   Q1 vs Q5        1
5                           num_cpc_code  8.293950e-03   Q1 vs Q5        1
6                          num_inventors -7.982152e-03   Q1 vs Q5        1
7                       num_words_5cate1  2.604949e-02   Q1 vs Q5        1
8                       num_words_5cate2 -1.251876e-02   Q1 vs Q5        1
9                       num_words_5cate3 -1.197335e-02   Q1 vs Q5        1
10                      num_words_5cate4 -6.296090e-03   Q1 vs Q5        1
11                      num_words_5cate5 -2.031151e-04   Q1 vs Q5        1
12        examiner_num_prior_apps_5cate1 -3.632539e-03   Q1 vs Q5        1
13        examiner_num_pr

## transfer

In [9]:
psm_covariates_formula_transfer <- as.formula("is_Q5 ~ flesch_reading_ease + concreteness + 
                                     novelty_min_qnorm + num_cpc_code + num_inventors + 
                                     num_words_5cate + examiner_num_prior_apps_5cate + 
                                     num_prior_apps_log + num_prior_patents_log + 
                                     attorney_num_prior_apps_log + 
                                     examiner_num_prior_apps_log +    
                                     attorney_num_prior_patents_log + 
                                     team_gender_n + attorney_gender_n + examiner_gender_n + 
                                     year + cpc_class_n + applicant_entity_n + business_entity_status_n")

### PSM

In [10]:
results_table <- data.frame()
comparison_groups <- c(1, 2, 3, 4)

for (comp_group in comparison_groups) {
  
  message(paste0("--------------------------------------------------"))
  message(paste0("Computing: Q", comp_group, " (Control) vs Q5 (Treated)..."))

  df_sub <- df_clean %>%
    filter(hype_quintile %in% c(comp_group, 5)) %>%
    mutate(
      is_Q5 = ifelse(hype_quintile == 5, 1, 0)
    )

  m.out <- matchit(psm_covariates_formula,
                   data = df_sub,
                   method = "nearest", 
                   ratio = 1,
                   replace = FALSE)
  
  df_matched <- match.data(m.out)
  q5_outcomes <- df_matched$application_transfered[df_matched$is_Q5 == 1]
  control_outcomes <- df_matched$application_transfered[df_matched$is_Q5 == 0]

  t_res <- t.test(x = q5_outcomes, y = control_outcomes)
  
  diff_val <- t_res$estimate[2] - t_res$estimate[1]
  ci_lower_flipped <- t_res$conf.int[2] * -1
  ci_upper_flipped <- t_res$conf.int[1] * -1

  tidy_res <- data.frame(
    term = "is_Q5",
    estimate = diff_val,              
    std.error = NA,                   
    p.value = t_res$p.value,          

    conf.low = ci_lower_flipped,      
    conf.high = ci_upper_flipped,     
    
    comparison = paste0("Q", comp_group, " vs Q5"),
    group_id = comp_group,
    mean_Q5 = t_res$estimate[1],
    mean_Control = t_res$estimate[2]
  )
  
  results_table <- bind_rows(results_table, tidy_res)
}

print(results_table %>% select(comparison, estimate, p.value, conf.low, conf.high))

--------------------------------------------------

Computing: Q1 (Control) vs Q5 (Treated)...

Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
--------------------------------------------------

Computing: Q2 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q3 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q4 (Control) vs Q5 (Treated)...



              comparison   estimate p.value   conf.low  conf.high
mean of y...1   Q1 vs Q5 0.05930136       0 0.05804454 0.06055819
mean of y...2   Q2 vs Q5 0.04635267       0 0.04506792 0.04763741
mean of y...3   Q3 vs Q5 0.04049947       0 0.03920260 0.04179633
mean of y...4   Q4 vs Q5 0.03083205       0 0.02951580 0.03214830


### Balance Check

In [ ]:
balance_table <- data.frame()
comparison_groups <- c(1, 2, 3, 4)

for (comp_group in comparison_groups) {
  
  message(paste0("--------------------------------------------------"))
  message(paste0("Computing: Q", comp_group, " (Control) vs Q5 (Treated)..."))

  df_sub <- df_clean %>%
    filter(hype_quintile %in% c(comp_group, 5)) %>%
    mutate(
      is_Q5 = ifelse(hype_quintile == 5, 1, 0)
    )

  m.out <- matchit(psm_covariates_formula_transfer,
                   data = df_sub,
                   method = "nearest", 
                   distance = "glm",  
                   caliper = 0.05,    
                   ratio = 1,
                   replace = FALSE)

  balance_summary <- summary(m.out, un = TRUE) 

  tidy_balance <- as.data.frame(balance_summary$sum.matched) %>%
    rownames_to_column(var = "covariate") %>%               
    select(covariate, matched_SMD = `Std. Mean Diff.`) %>%
    mutate(
      comparison = paste0("Q", comp_group, " vs Q5"),     
      group_id = comp_group                                 
    )
  balance_table <- bind_rows(balance_table, tidy_balance)
}
print(balance_table)

--------------------------------------------------

Computing: Q1 (Control) vs Q5 (Treated)...

Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
--------------------------------------------------

Computing: Q2 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q3 (Control) vs Q5 (Treated)...

--------------------------------------------------

Computing: Q4 (Control) vs Q5 (Treated)...



                               covariate   matched_SMD comparison group_id
1                               distance  4.097209e-02   Q1 vs Q5        1
2                    flesch_reading_ease -2.310652e-02   Q1 vs Q5        1
3                           concreteness  5.044224e-03   Q1 vs Q5        1
4                      novelty_min_qnorm -2.346193e-02   Q1 vs Q5        1
5                           num_cpc_code  6.596601e-03   Q1 vs Q5        1
6                          num_inventors -6.520435e-03   Q1 vs Q5        1
7                       num_words_5cate1  2.503504e-02   Q1 vs Q5        1
8                       num_words_5cate2 -1.467497e-02   Q1 vs Q5        1
9                       num_words_5cate3 -1.277898e-02   Q1 vs Q5        1
10                      num_words_5cate4 -2.506150e-03   Q1 vs Q5        1
11                      num_words_5cate5  5.356463e-04   Q1 vs Q5        1
12        examiner_num_prior_apps_5cate1 -3.301831e-03   Q1 vs Q5        1
13        examiner_num_pr